# CNM Blog Post – Figures

Publikationsreife Plots für den Blog-Eintrag zum Projekt
**Cluster-Based Network Model (CNM) for 3D Cylinder Wake at Re ≈ 3900**.

**Workflow:**
1. Run `python/scripts/run_cnm_analysis.py` first → produces `.npy` outputs
2. Set `DATA_DIR` below to that output folder
3. Run all cells → figures saved to `FIGURE_DIR`

A synthetic demo dataset is generated automatically if the real data is not yet available.

In [ ]:
from pathlib import Path

# ── CONFIGURE THESE ─────────────────────────────────────────────────────────
# Folder produced by run_cnm_analysis.py (contains .npy files + summary.json)
DATA_DIR = Path("../../data/figures/cnm_force")

# Where to save blog figures
FIGURE_DIR = Path("../../data/figures/blog")

# Force coefficients file (for Block 1)
FORCE_FILE = Path("../../openfoam/cases/cylinderLES3D/postProcessing/forceCoeffs/0/coefficient.dat")
# ─────────────────────────────────────────────────────────────────────────────

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data dir  : {DATA_DIR}")
print(f"Figure dir: {FIGURE_DIR}")
print(f"Data exists: {DATA_DIR.exists()}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.ticker import MaxNLocator
from mpl_toolkits.mplot3d import Axes3D
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# ── Global plot style ────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "font.family":        "sans-serif",
    "font.size":          11,
    "axes.titlesize":     12,
    "axes.labelsize":     11,
    "legend.fontsize":    10,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "lines.linewidth":    1.5,
    "savefig.bbox":       "tight",
    "savefig.format":     "png",
})

# Consistent colour palette across all plots
C_DATA   = "#2c7bb6"   # blue  – original data
C_CNM    = "#d7191c"   # red   – CNM prediction
C_ACCENT = "#fdae61"   # amber – highlights

CMAP_CLUSTER = plt.cm.tab10
CMAP_MATRIX  = "viridis"

def save(fig, name, also_svg=True):
    """Save figure as PNG (and optionally SVG) to FIGURE_DIR."""
    fig.savefig(FIGURE_DIR / f"{name}.png")
    if also_svg:
        fig.savefig(FIGURE_DIR / f"{name}.svg")
    print(f"Saved: {name}")

print("Style loaded.")

In [ ]:
# ── Load data (real or synthetic demo) ──────────────────────────────────────

def make_demo_data(K=8, n=300, n_modes=4, seed=42):
    """Generate synthetic cylinder-like data for notebook development."""
    rng = np.random.default_rng(seed)
    t   = np.linspace(0, 60, n)
    St  = 0.21          # Strouhal number Re=3900
    f   = St            # non-dim shedding frequency (U=D=1)

    # Synthetic force coefficients
    Cd = 1.05 + 0.05 * np.sin(4 * np.pi * f * t) + 0.01 * rng.standard_normal(n)
    Cl = 0.50 * np.sin(2 * np.pi * f * t + 0.1)  + 0.05 * rng.standard_normal(n)
    Cm = 0.05 * np.cos(2 * np.pi * f * t)         + 0.005 * rng.standard_normal(n)

    # Synthetic POD coefficients (paired sine/cosine modes)
    A = np.column_stack([
        0.08 * np.sin(2 * np.pi * f * t),
        0.08 * np.cos(2 * np.pi * f * t),
        0.03 * np.sin(4 * np.pi * f * t),
        0.03 * np.cos(4 * np.pi * f * t),
    ])
    A += 0.003 * rng.standard_normal(A.shape)

    # Synthetic singular values
    sv = np.array([12.0, 11.8, 4.1, 4.0, 1.2, 0.8, 0.4, 0.2,
                   0.1, 0.05, 0.02, 0.01])
    energy     = sv**2 / (sv**2).sum()
    cum_energy = np.cumsum(energy)

    # Synthetic cluster labels (periodic sequence with small noise)
    raw_labels = (np.floor(K * (0.5 + 0.5 * np.sin(2 * np.pi * f * t)))).astype(int)
    raw_labels = np.clip(raw_labels, 0, K - 1)
    labels     = raw_labels.copy()
    # inject some random cluster jumps for realism
    jump_idx   = rng.integers(0, n, size=20)
    labels[jump_idx] = rng.integers(0, K, size=20)

    # Synthetic transition matrices
    P = np.zeros((K, K))
    for i in range(len(labels) - 1):
        P[labels[i+1], labels[i]] += 1
    col = P.sum(axis=0, keepdims=True)
    P   = np.divide(P, col, out=np.zeros_like(P), where=col > 0)

    Q = P.copy()
    np.fill_diagonal(Q, 0)
    col = Q.sum(axis=0, keepdims=True)
    Q   = np.divide(Q, col, out=np.zeros_like(Q), where=col > 0)

    T = np.full((K, K), np.nan)
    for j in range(K):
        for k in range(K):
            if j != k and Q[j, k] > 0:
                T[j, k] = 1.0 / (f + 0.05 * rng.standard_normal())

    # Synthetic CNM prediction (phase-shifted version of labels)
    shift       = int(0.15 * n / K)
    pred_labels = np.roll(labels, shift)
    pred_A      = np.roll(A, shift, axis=0)

    centers = np.array([
        [0.08 * np.sin(2*np.pi*k/K), 0.08 * np.cos(2*np.pi*k/K),
         0.03 * np.sin(4*np.pi*k/K), 0.03 * np.cos(4*np.pi*k/K)]
        for k in range(K)
    ])

    return dict(
        t=t, Cd=Cd, Cl=Cl, Cm=Cm,
        A=A, sv=sv, energy=energy, cum_energy=cum_energy, n_modes=4,
        labels=labels, pred_labels=pred_labels,
        pred_A=pred_A, centers=centers,
        P=P, Q=Q, T=T, K=K,
    )


def load_npy(folder: Path):
    """Load .npy outputs from run_cnm_analysis.py."""
    d = {}
    d["t"]           = np.load(folder / "time.npy")
    d["A"]           = np.load(folder / "pod_coefficients.npy")
    d["labels"]      = np.load(folder / "cluster_labels.npy")
    d["centers"]     = np.load(folder / "cluster_centers.npy")
    d["P"]           = np.load(folder / "P_matrix.npy")
    d["Q"]           = np.load(folder / "Q_matrix.npy")
    d["T"]           = np.load(folder / "T_matrix.npy")
    d["K"]           = int(d["P"].shape[0])

    if (folder / "cnm_pred_labels.npy").exists():
        d["pred_labels"] = np.load(folder / "cnm_pred_labels.npy")
        d["pred_A"]      = np.load(folder / "cnm_pred_coefficients.npy")
    else:
        d["pred_labels"] = None
        d["pred_A"]      = None

    with open(folder / "summary.json") as f:
        summary = json.load(f)
    d["n_modes"]     = summary["pod_n_modes"]
    d["cum_energy"]  = None   # recompute if needed
    d["energy"]      = None
    d["sv"]          = None
    d["Cd"] = d["Cl"] = d["Cm"] = None   # loaded separately in Block 1
    return d


def load_force_coeffs(path: Path):
    """Parse OpenFOAM forceCoeffs.dat → arrays."""
    rows, header = [], None
    with open(path) as f:
        for line in f:
            s = line.strip()
            if not s: continue
            if s.startswith("#"):
                cands = s.lstrip("#").split()
                if cands and cands[0].lower() == "time":
                    header = cands
                continue
            try:
                rows.append([float(x) for x in s.split()])
            except ValueError:
                pass
    arr = np.array(rows)
    fallback = ["Time","Cm","Cd","Cl","Cl_f","Cl_r"]
    if header is None or len(header) != arr.shape[1]:
        header = fallback[:arr.shape[1]]
    idx = {h: i for i, h in enumerate(header)}
    return {
        "t":  arr[:, idx["Time"]],
        "Cd": arr[:, idx.get("Cd", 2)],
        "Cl": arr[:, idx.get("Cl", 3)],
        "Cm": arr[:, idx.get("Cm", 1)],
    }


# ── Decide: real data or demo ────────────────────────────────────────────────
if DATA_DIR.exists() and (DATA_DIR / "summary.json").exists():
    print("Loading real data from", DATA_DIR)
    D = load_npy(DATA_DIR)
    if FORCE_FILE.exists():
        fc = load_force_coeffs(FORCE_FILE)
        D.update(fc)
    else:
        print(f"  Force file not found ({FORCE_FILE}), skipping Block 1 raw plot.")
    USE_DEMO = False
else:
    print("Real data not found — using synthetic demo dataset.")
    D = make_demo_data(K=8, n=300)
    USE_DEMO = True

print(f"  t range : {D['t'].min():.1f} – {D['t'].max():.1f}")
print(f"  K       : {D['K']}")
print(f"  n_modes : {D['n_modes']}")
print(f"  Demo    : {USE_DEMO}")

---
## Block 1 – Simulation Overview: Force Coefficients & Strouhal Frequency

In [ ]:
from scipy.signal import welch

t  = D["t"]
Cd = D["Cd"]
Cl = D["Cl"]
Cm = D["Cm"]

fig = plt.figure(figsize=(12, 7))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.32)

# ── (a) Cd time series ───────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(t, Cd, color=C_DATA, lw=1.4)
ax0.set_xlabel("Time  $t \\cdot U_\\infty / D$")
ax0.set_ylabel("$C_D$")
ax0.set_title("(a) Drag coefficient")

# ── (b) Cl time series ───────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
ax1.plot(t, Cl, color=C_ACCENT, lw=1.4)
ax1.set_xlabel("Time  $t \\cdot U_\\infty / D$")
ax1.set_ylabel("$C_L$")
ax1.set_title("(b) Lift coefficient")

# ── (c) Cl PSD → Strouhal number ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
dt  = float(np.median(np.diff(t)))
fs  = 1.0 / dt
# Use only settled part (skip first 20% as transient)
i0  = int(0.2 * len(Cl))
nperseg = min(256, (len(Cl) - i0) // 2)
f_psd, Pxx = welch(Cl[i0:], fs=fs, nperseg=nperseg)
f_psd_nondim = f_psd  # already St = f*D/U with D=U=1
ax2.semilogy(f_psd_nondim, Pxx, color=C_DATA, lw=1.4)
peak_idx = np.argmax(Pxx)
St_peak  = f_psd_nondim[peak_idx]
ax2.axvline(St_peak, color=C_CNM, ls="--", lw=1.2,
            label=f"$St = {St_peak:.3f}$")
ax2.set_xlabel("Strouhal number  $St = f D / U_\\infty$")
ax2.set_ylabel("PSD")
ax2.set_title("(c) Lift PSD – Strouhal frequency")
ax2.set_xlim(0, min(2.0, f_psd_nondim[-1]))
ax2.legend()

# ── (d) Cd vs Cl phase portrait ───────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
sc  = ax3.scatter(Cd[i0:], Cl[i0:], c=t[i0:], cmap="plasma",
                  s=3, alpha=0.6, linewidths=0)
plt.colorbar(sc, ax=ax3, label="Time")
ax3.set_xlabel("$C_D$")
ax3.set_ylabel("$C_L$")
ax3.set_title("(d) Phase portrait $C_D$ – $C_L$")

fig.suptitle("Re ≈ 3 900 — 3D LES Cylinder Wake: Force Coefficients",
             fontsize=13, fontweight="bold", y=1.01)
save(fig, "01_force_overview")
plt.show()

---
## Block 2 – POD / SVD: Energy Spectrum & Mode Coefficients

In [ ]:
A       = D["A"]
n_modes = D["n_modes"]

# Recompute energy from POD coefficients if not loaded from file
if D["sv"] is None:
    # Reconstruct approximate singular values from coefficient RMS
    sv_approx  = np.sqrt((A**2).mean(axis=0))
    energy     = sv_approx**2 / (sv_approx**2).sum()
    cum_energy = np.cumsum(energy)
else:
    sv_approx  = D["sv"]
    energy     = D["energy"]
    cum_energy = D["cum_energy"]

n_show = min(len(energy), 20)
x_idx  = np.arange(1, n_show + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# ── (a) Energy spectrum ───────────────────────────────────────────────────────
ax = axes[0]
ax.bar(x_idx, energy[:n_show] * 100, color=C_DATA, alpha=0.8,
       label="Individual")
ax2 = ax.twinx()
ax2.plot(x_idx, cum_energy[:n_show] * 100, "o-",
         color=C_CNM, lw=1.5, ms=5, label="Cumulative")
ax2.axhline(99.99, color="grey", ls="--", lw=1.0, label="99.99 %")
ax.axvline(n_modes + 0.5, color="black", ls=":", lw=1.2,
           label=f"Kept: {n_modes} modes")
ax.set_xlabel("POD mode index")
ax.set_ylabel("Individual energy  [%]", color=C_DATA)
ax2.set_ylabel("Cumulative energy  [%]", color=C_CNM)
ax.set_title("(a) POD energy spectrum")
ax.set_xlim(0.5, n_show + 0.5)
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
lines1, labs1 = ax.get_legend_handles_labels()
lines2, labs2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labs1 + labs2, loc="upper right", fontsize=9)

# ── (b) First 4 mode coefficients over time ───────────────────────────────────
ax = axes[1]
n_plot = min(4, A.shape[1])
colors = [plt.cm.Set1(i) for i in range(n_plot)]
for i in range(n_plot):
    ax.plot(t, A[:, i], color=colors[i], lw=1.2,
            label=f"$a_{{{i+1}}}$", alpha=0.85)
ax.set_xlabel("Time  $t \\cdot U_\\infty / D$")
ax.set_ylabel("POD coefficient")
ax.set_title("(b) First 4 POD mode coefficients")
ax.legend(ncol=2)

fig.suptitle("Snapshot POD / SVD — Cylinder Wake",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "02_pod_energy")
plt.show()

---
## Block 3 – Clustering & MDS: State Space Structure

In [ ]:
from sklearn.manifold import MDS

labels  = D["labels"]
centers = D["centers"]
K       = D["K"]
P, Q, T = D["P"], D["Q"], D["T"]

# ── Recompute MDS embedding ──────────────────────────────────────────────────
combined = np.vstack([A, centers])
n_comp   = min(3, combined.shape[0] - 1, combined.shape[1])
n_comp   = max(2, n_comp)

mds = MDS(n_components=n_comp, random_state=42, metric_mds=True,
          n_init=4, init="random", max_iter=300)
emb = mds.fit_transform(combined)
if emb.shape[1] == 2:
    emb = np.c_[emb, np.zeros(len(emb))]

emb_A = emb[:len(A)]
emb_C = emb[len(A):]

fig = plt.figure(figsize=(15, 5))
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# ── (a) 3D MDS cluster map ────────────────────────────────────────────────────
ax3d = fig.add_subplot(gs[0], projection="3d")
sc   = ax3d.scatter(emb_A[:, 0], emb_A[:, 1], emb_A[:, 2],
                    c=labels, cmap=CMAP_CLUSTER, vmin=0, vmax=K-1,
                    s=8, alpha=0.7, linewidths=0)
ax3d.scatter(emb_C[:, 0], emb_C[:, 1], emb_C[:, 2],
             s=120, marker="X", c="black", zorder=5)
for k, c in enumerate(emb_C):
    ax3d.text(c[0], c[1], c[2], f" {k}", fontsize=8)
ax3d.set_title("(a) 3D MDS proximity map")
ax3d.set_xlabel("$x_1$"); ax3d.set_ylabel("$x_2$"); ax3d.set_zlabel("$x_3$")
fig.colorbar(sc, ax=ax3d, label="Cluster", shrink=0.6)

# ── (b) 2D MDS projection x1-x2 ──────────────────────────────────────────────
ax2d = fig.add_subplot(gs[1])
sc2  = ax2d.scatter(emb_A[:, 0], emb_A[:, 1],
                    c=labels, cmap=CMAP_CLUSTER, vmin=0, vmax=K-1,
                    s=10, alpha=0.7, linewidths=0)
ax2d.scatter(emb_C[:, 0], emb_C[:, 1], s=120, marker="X",
             c="black", zorder=5)
for k, c in enumerate(emb_C):
    ax2d.annotate(f"{k}", (c[0], c[1]), fontsize=9,
                  xytext=(4, 4), textcoords="offset points")
ax2d.set_xlabel("$x_1$"); ax2d.set_ylabel("$x_2$")
ax2d.set_title("(b) $x_1$–$x_2$ projection")
fig.colorbar(sc2, ax=ax2d, label="Cluster", shrink=0.8)

# ── (c) CMM transition probability matrix Q ──────────────────────────────────
axQ  = fig.add_subplot(gs[2])
imQ  = axQ.imshow(Q, origin="lower", cmap=CMAP_MATRIX, aspect="equal",
                  vmin=0, vmax=Q.max() if Q.max() > 0 else 1)
plt.colorbar(imQ, ax=axQ, label="Transition probability")
axQ.set_xlabel("From cluster $k$")
axQ.set_ylabel("To cluster $j$")
axQ.set_title("(c) CNM matrix $\\mathbf{Q}$")
axQ.xaxis.set_major_locator(MaxNLocator(integer=True))
axQ.yaxis.set_major_locator(MaxNLocator(integer=True))

fig.suptitle("k-means++ Clustering & State-Space Structure",
             fontsize=13, fontweight="bold")
save(fig, "03_clustering_mds")
plt.show()

In [ ]:
# ── P / Q / T matrix triple plot (as in Sphere report) ───────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
titles  = ["$\\mathbf{P}$ — CMM (incl. self)",
           "$\\mathbf{Q}$ — CNM (inter-cluster)",
           "$\\mathbf{T}$ — Transition time"]
matrices = [P, Q, T]
labels_cb = ["probability", "probability", "time units"]

for ax, M, title, cblabel in zip(axes, matrices, titles, labels_cb):
    finite_vals = M[np.isfinite(M)]
    vmax = finite_vals.max() if finite_vals.size > 0 else 1.0
    im = ax.imshow(np.where(np.isfinite(M), M, 0),
                   origin="lower", cmap=CMAP_MATRIX,
                   aspect="equal", vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax, label=cblabel, fraction=0.046)
    ax.set_title(title)
    ax.set_xlabel("From $k$")
    ax.set_ylabel("To $j$")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

fig.suptitle("CMM/CNM Transition Matrices",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "04_transition_matrices")
plt.show()

---
## Block 4 – CNM Reconstruction vs. Data

In [ ]:
pred_labels = D["pred_labels"]
pred_A      = D["pred_A"]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

# ── (a) Cluster affiliation over time ────────────────────────────────────────
ax = axes[0]
ax.step(t, labels, where="post", color=C_DATA, lw=1.4, label="Data")
if pred_labels is not None:
    ax.step(t, pred_labels, where="post", color=C_CNM, lw=1.2,
            ls="--", alpha=0.85, label="CNM prediction")
ax.set_ylabel("Cluster index")
ax.set_title("(a) Temporal evolution of cluster affiliation")
ax.legend()
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

# ── (b) POD coefficients: data vs CNM ────────────────────────────────────────
ax = axes[1]
n_plot  = min(4, A.shape[1])
palette = [plt.cm.Set1(i) for i in range(n_plot)]
for i in range(n_plot):
    ax.plot(t, A[:, i], color=palette[i], lw=1.3, alpha=0.9,
            label=f"$a_{{{i+1}}}$ data")
    if pred_A is not None:
        ax.plot(t, pred_A[:, i], color=palette[i], lw=1.1,
                ls="--", alpha=0.7, label=f"$a_{{{i+1}}}$ CNM")
ax.set_xlabel("Time  $t \\cdot U_\\infty / D$")
ax.set_ylabel("POD coefficient")
ax.set_title("(b) First 4 POD coefficients — data vs. CNM reconstruction")
ax.legend(ncol=n_plot, loc="upper right", fontsize=9)

fig.suptitle("CNM Reconstruction vs. Simulation Data",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "05_cnm_reconstruction")
plt.show()

In [ ]:
# ── Cluster population: data vs CNM ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4.5))

x        = np.arange(K)
data_pop = np.bincount(labels, minlength=K) / len(labels)

if pred_labels is not None:
    pred_pop = np.bincount(pred_labels.astype(int), minlength=K) / len(pred_labels)
    w = 0.38
    ax.bar(x - w/2, data_pop * 100, width=w, color=C_DATA,
           alpha=0.85, label="Data")
    ax.bar(x + w/2, pred_pop * 100, width=w, color=C_CNM,
           alpha=0.75, label="CNM")
else:
    ax.bar(x, data_pop * 100, width=0.6, color=C_DATA, label="Data")

ax.set_xlabel("Cluster index")
ax.set_ylabel("Occupation probability  [%]")
ax.set_title("Cluster probability distribution — data vs. CNM")
ax.set_xticks(x)
ax.legend()

fig.tight_layout()
save(fig, "06_cluster_probabilities")
plt.show()

---
## Block 5 – Blog Export Summary

In [ ]:
# ── List all generated figures ────────────────────────────────────────────────
figures = sorted(FIGURE_DIR.glob("*.png"))

print(f"{'='*55}")
print(f"Blog figures written to: {FIGURE_DIR}")
print(f"{'='*55}")
for p in figures:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<40}  {size_kb:6.0f} kB")

print()
print("Suggested blog figure order:")
blog_order = [
    ("01_force_overview",       "LES validation – Cd/Cl time series + Strouhal FFT"),
    ("02_pod_energy",           "POD energy spectrum – how many modes are needed?"),
    ("03_clustering_mds",       "State-space structure – MDS proximity map"),
    ("04_transition_matrices",  "CMM/CNM matrices P, Q, T"),
    ("05_cnm_reconstruction",   "CNM vs. data: cluster affiliation + POD coefficients"),
    ("06_cluster_probabilities","Cluster occupation – data vs. CNM"),
]
for fname, caption in blog_order:
    print(f"  {fname+'.png':<40}  → {caption}")